# The Kronecker construction for multinomial models

This chapter records the construction and reconstruction of R `mboost`'s
`%O%` (Kronecker) operator, and the per-class penalty structure it implies.

## Response and gradient

A $K$-class response is dummy-coded into $K-1$ columns (reference class dropped)
and stored **row-major** as a length-$n(K-1)$ vector
$y[i(K{-}1)+k]$. The engine derives the observation count $n$ from the weights
(not from `y`'s length), so the same loop handles the expanded response. The
softmax, loss, and gradient are as in [](../methodology/multinomial).

```{admonition} Softmax stabilization
:class: warning
The stabilized softmax uses $m_i = \max(0, \max_k f_{ik})$. The reference class's
implicit logit is 0, so its denominator term is $e^{0-m_i} = e^{-m_i}$.
`mboostrs` uses this term in `negative_gradient`, `risk`,
`ngradient_and_risk` and `response`. R `mboost` instead bounds its predictors
before exponentiating and uses an unshifted denominator, where the reference
term is correctly 1. The risk path agrees with R to ~1e-13; see
[](parity_and_bugs.ipynb) for the verified distinction.
```

## Sharing a learner across classes

For a covariate effect with covariate design $X_1$ ($n\times p_1$) and class design
$X_2 = I_{K-1}$, the combined design is $X_2 \otimes X_1$ over the stacked
response. With $X_2 = I$ the penalized normal equations are **block-diagonal**, so
the fit is $K-1$ independent ridge solves sharing a single factorization of
$X_1^\top W X_1 + P$. `KroneckerBols` stores $X_1$, $X_2$, the penalty $P$, and one
`GramSolver`, and assembles $\operatorname{vec}(X_1\beta X_2^\top)$ as the
prediction.

Because this is the dominant Multinomial case, the implementation records
whether $X_2$ is exactly the identity. When it is, fit-time prediction,
`predict_into`, `LearnerStore.predict` (`LearnerStore` holds the stored learners of a fitted model), and CV out-of-bag risk evaluation use
direct per-class dot products
$(X_1\beta_k)_i$ instead of the general $X_1 B X_2^\top$ reconstruction. The
result is identical, but avoids an extra nested loop over the class-design
columns.

### Explicit example: 3 classes × 2 covariates

A 3-class multinomial ($K=3$, so $K-1=2$ class columns) with two scalar
covariates $x_1$, $x_2$ and a per-class intercept produces the Kronecker design
$X_2 \otimes X_1$:

$$
\underset{(2n\,\times\,6)}{X_{\text{kronecker}}} \;=\;
\underset{(2\times 2)}{I_2} \otimes
\underset{(n\times 3)}{\begin{pmatrix}1 & x_{11} & x_{12}\\ \vdots & \vdots & \vdots\\1 & x_{n1} & x_{n2}\end{pmatrix}}
\;=\;
\begin{pmatrix}
  X_1 & 0   \\
  0   & X_1
\end{pmatrix}
$$

where the $6$ columns correspond to $(\beta_{0,1},\beta_{1,1},\beta_{2,1},
 \beta_{0,2},\beta_{1,2},\beta_{2,2})$ — intercept and two slopes for each of
the $K-1=2$ logit classes. If the covariate effect is smooth (`bbs(x, df=4)`
instead of `bols`), $X_1$ is the $n\times 24$ B-spline basis and the block
structure is identical.

The per-class penalty matrix is $\lambda(I_{24} + K_1)$:
the bbs difference penalty $K_1$ plus a ridge identity from the
`bols(class, df=K-1)` reference design. The `+I` term matches R's `%O%` operator; without it, the smooth term's
penalty has no lower bound and the per-class df is systematically wrong.

The block-diagonal Gram $G_b = X_1^\top W X_1 + \lambda(I + K_1)$ for
class $b$ has the explicit structure:

$$
G_b = \lambda I + \underbrace{X_1^\top W X_1}_{\text{marginal Gram}} + \lambda \underbrace{K_1}_{\text{diff penalty}}.
$$

Written out for $p_1 = 3$ with a first-order difference penalty
($K_1 = D^\top D$ where $D$ is $(p_1-1)\times p_1$; the `bbs` default is a second-order penalty, so this is the first-order case):

$$
G_b = \begin{pmatrix}
  g_{11} + \lambda(1 + 1) & g_{12} - \lambda           & g_{13} \\
  g_{12} - \lambda         & g_{22} + \lambda(1 + 2)   & g_{23} - \lambda \\
  g_{13}                   & g_{23} - \lambda           & g_{33} + \lambda(1 + 1)
\end{pmatrix},
$$

where $g_{jk}$ is entry $(j,k)$ of $X_1^\top W X_1$. The `+I` contributes
$\lambda$ down the diagonal; $K_1$ adds $+2\lambda$ on the interior diagonal
and $-\lambda$ off-diagonal. Without the `+I` term, the penalty could vanish
for coefficients in the null space of $D$ (constant vectors).

## R's penalty for `bbs %O% bols(class)`

For a **linear** (`bols`) effect, $P = 0$ and the per-class fit is plain OLS, which agrees with R. For a **smooth** (`bbs`) effect, R's penalty structure, verified against `mboost` on `iris`, is summarized as follows:

1. **Block structure is per-class.** The extracted Kronecker Gram and penalty are
   exactly $I_{K-1}\otimes(\cdot)$ — classes are independent given the shared
   design. So the engine's per-class solve is structurally right.
2. **The combined df is the product.** R's `%O%` targets
   $\mathrm{df} = \mathrm{df}_1 \cdot \mathrm{df}_2$ for the full Kronecker design;
   over the $K-1$ identical class blocks this is a per-class target of
   $\mathrm{df}_1$.
3. **The penalty is $I + K_1$, not $K_1$.** R's `%O%` forms the anisotropic
   penalty $K_2 \otimes I_{p_1} + I_{p_2}\otimes K_1$. With the standard
   `bols(class, df = K-1, contr.dummy)` learner (`contr.dummy` is R's dummy-coding contrast, which gives each class level its own indicator column) $K_2 = I$, so the per-class penalty
   is the bbs **difference penalty plus a ridge identity**, $\lambda(I + K_1)$.
   This $+I$ term — the `bols`-class ridge entering the smooth penalty — cannot
   be recovered by adjusting `λ` or centering alone.

The recipe, implemented in `prepare_kronecker_bbs`:

> design = raw (un-centered) B-spline basis $X_1$;
> penalty = $I + K_1$ where $K_1$ is the bbs difference penalty;
> $\lambda$ solved so the per-class effective df equals $\mathrm{df}_1$ on that
> penalty; $X_2 = I_{K-1}$.

It reproduces R's fitted values to $\sim 10^{-11}$ in isolation and $\sim 10^{-6}$
end-to-end, and is guarded by an R-independent core test that asserts the per-class
ridge uses $(I + K_1)$.

## Prediction on new data

In-sample prediction uses the stored Kronecker learner directly. For new data, the
front-end rebuilds each learner's $X_1$ at the new covariates — `[1,x]`/`[x]` for
`bols`, and the raw B-spline basis at the new $x$ using the **training** boundary
knots (exposed by a small `bbs_raw_design` helper) for `bbs` — then applies the
stored coefficient reshaped to $(K-1, p_1)$ class blocks. This was validated to
machine precision by checking that predicting on the training covariates
reproduces the in-sample fit.

## Unsupported learners

Only `bols` and `bbs` learners are Kronecker-wrappable here; `btree`/`bmono`/etc.
raise a clear error. AIC is unavailable for the multinomial model — R cannot
compute it either, since the `%O%` learner has no hat matrix in `mboost`.

## See also

- [](linear_algebra.ipynb) — the `GramSolver` used by the Kronecker per-class solve.
- [](../methodology/families) — the family trait and how multinomial integrates into it.
- [](../methodology/multinomial) — the multinomial loss, softmax, and gradient.